# LLM Eval System — one-shot real run on Colab free T4

End-to-end: installs the project, boots vLLM on a small open-weight model,
runs Parts B / C / D / E to produce **real** numbers, zips everything up, and
triggers a browser download so you can unpack the artefacts into your repo
and commit them.

## Before you run this

1. **Push your fork first** — the install cell clones from `REPO_URL`.
   If your fork is behind, the cell fails with a clear 'push your fix' message.
2. `Runtime` → `Change runtime type` → **T4 GPU**.
3. Edit the first code cell:
   - `REPO_URL` — your GitHub URL.
   - `MODEL` — default is `Qwen/Qwen2.5-1.5B-Instruct` (ungated, no HF token
     needed). Alternatives:
     - `microsoft/Phi-3-mini-4k-instruct` (3.8 B, ungated, higher accuracy)
     - `meta-llama/Llama-3.2-1B-Instruct` (gated — set `HF_TOKEN`)
4. `Runtime` → `Run all`. Wall time ~45–60 min. **Keep the tab visible**
   (Colab idle-kills a backgrounded tab after ~20 min).

## Built-in safety gates

This notebook is hardened against the two failure modes that bit earlier:

- **Stale-clone trap.** The install cell force-deletes any pre-existing
  `/content/LLM-Evaluation-Pipeline` before cloning, so re-runs in the same Colab
  session can't keep an old HEAD alive.
- **Synthetic-data trap.** The install cell asserts `improve/eval.sh`
  contains the 'Always re-prepare' fix (commit `bacff58`+); the Part E cell
  asserts the first HellaSwag row's `ind` is numeric (not `syn-*`) and that
  no variant has accuracy exactly 0.0 or 1.0 (the synthetic-data fingerprint);
  the packaging cell re-checks both before producing the tarball.

If any gate trips, the run fails loudly rather than producing a bogus
tarball. You cannot accidentally submit synthetic numbers.

The last cell downloads `llmeval_results.tgz` to your laptop. Unpack it
into your repo checkout, commit, push — you're done.

## 1. Config — edit these

In [ ]:
REPO_URL = "https://github.com/harryboi17/LLM-Evaluation-Pipeline.git"
BRANCH = "main"

# Pick a model. Un-gated defaults avoid HF_TOKEN friction.
# For Llama-3.2, set HF_TOKEN = "hf_xxxx" below and accept the license first
# at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

HF_TOKEN = ""  # only needed for gated models

# How many examples per task. 200 gives tight CIs in ~10-20 min;
# 500 is ideal for a final report; use 50 for fast iteration.
N_EVAL = 200

# vLLM serving args — safe defaults for a T4.
MAX_MODEL_LEN = 2048
DTYPE = "float16"            # T4 is Turing (SM 7.5) -- no native bf16. Use 'bfloat16' only on A100/H100.
GPU_MEM_UTIL = 0.85

## 2. Confirm we actually have a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 3. Install uv + clone the project

In [ ]:
import os, shutil, subprocess

# uv (fast Python dep manager). Persist in PATH for later cells.
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
!uv --version

# Force a fresh clone every time. Without this, re-running the notebook in
# a previously-connected Colab session reuses the old /content/LLM-Evaluation-Pipeline
# workdir (with any stale synthetic data still on disk and the old HEAD).
# This was the single biggest source of 'my Part E numbers look synthetic'
# while developing -- never again.
os.chdir("/content")
shutil.rmtree("LLM-Evaluation-Pipeline", ignore_errors=True)
!git clone --branch $BRANCH $REPO_URL LLM-Evaluation-Pipeline
os.chdir("/content/LLM-Evaluation-Pipeline")
!git log --oneline -1

# Gatekeep: the clone MUST contain the 'Always re-prepare' fix in
# improve/eval.sh. If it doesn't, the fork is behind -- abort now rather
# than 30 minutes from now with synthetic numbers.
from pathlib import Path
if "Always re-prepare" not in Path("improve/eval.sh").read_text():
    raise RuntimeError(
        "improve/eval.sh is missing the 'Always re-prepare' fix (commit bacff58+).\n"
        "Your fork at REPO_URL is behind. Push the latest main and re-run this cell."
    )
print("OK: eval.sh contains the fix")

# Full install -- all extras in one shot so torch/vllm/lm-eval land together.
# This takes ~5-10 min cold; cached across re-runs inside the same session.
!uv sync --extra serve --extra eval --extra perf --extra improve

## 4. (Optional) HuggingFace login for gated models

In [ ]:
if HF_TOKEN:
    !uv run huggingface-cli login --token $HF_TOKEN --add-to-git-credential
else:
    print("HF_TOKEN empty -- gated models (e.g. Llama-3.2) will fail to download.")
    print("If you're using a gated model, paste a token in the config cell and rerun.")

## 5. Configure Settings via env vars

In [ ]:
os.environ.update({
    "LLMEVAL_MOCK_BACKEND": "false",
    "LLMEVAL_MODEL_NAME": MODEL,
    "LLMEVAL_VLLM_HOST": "127.0.0.1",
    "LLMEVAL_VLLM_PORT": "8000",
    "LLMEVAL_VLLM_API_KEY": "EMPTY",
    "LLMEVAL_VLLM_DTYPE": DTYPE,
    "LLMEVAL_VLLM_MAX_MODEL_LEN": str(MAX_MODEL_LEN),
    "LLMEVAL_VLLM_GPU_MEMORY_UTILIZATION": str(GPU_MEM_UTIL),
    "LLMEVAL_VLLM_TIMEOUT_S": "300",
    "LLMEVAL_SEED": "42",
    "LLMEVAL_LOG_FORMAT": "json",
    # Colab-friendliness: force the spawn multiproc method so the worker
    # subprocesses don't inherit a live CUDA context from the kernel
    # (common cause of 'Engine core initialization failed' on free T4).
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
})

# Sanity: mock-backend test suite still works with the new settings.
!uv run pytest tests/common -q

## 6. Boot vLLM in the background and wait for it to be ready

The first run downloads the model weights from HuggingFace (one-time
~2-8 GB depending on the model). Subsequent runs in the same Colab session
are cached.

We don't use `make serve` here because we need the Python process handle
to kill it cleanly after the eval + perf runs.

In [ ]:
import subprocess, time, urllib.request, urllib.error

serve_log = open("vllm_serve.log", "w")
serve_proc = subprocess.Popen(
    ["uv", "run", "python", "-m", "serve.serve"],
    stdout=serve_log, stderr=subprocess.STDOUT,
)
print(f"vLLM pid={serve_proc.pid}, log=vllm_serve.log, tailing readiness ...")

# Poll /v1/models until it responds, then we're ready.
url = "http://127.0.0.1:8000/v1/models"
deadline = time.monotonic() + 900  # 15 min cap for the first weight download
ready = False
while time.monotonic() < deadline:
    if serve_proc.poll() is not None:
        print("!! vLLM exited early. Last 40 log lines:")
        !tail -n 40 vllm_serve.log
        raise RuntimeError("vLLM failed to start.")
    try:
        req = urllib.request.Request(url, headers={"Authorization": "Bearer EMPTY"})
        resp = urllib.request.urlopen(req, timeout=3)
        if resp.status == 200:
            ready = True
            break
    except (urllib.error.URLError, ConnectionRefusedError, TimeoutError):
        pass
    time.sleep(5)

if not ready:
    raise RuntimeError("vLLM didn't come up in 15 min. See vllm_serve.log.")
print("vLLM is serving. Moving on.")

## 7. Sanity: one quick completion to prove end-to-end

In [ ]:
!uv run python -m serve.client "What is the capital of France?" --max-tokens 16

## 8. Part B — lm-eval on MMLU-STEM + HellaSwag + custom_qa

In [ ]:
!uv run python -m eval_runner.run_eval \
    --task mmlu_stem,hellaswag,custom_qa \
    --limit $N_EVAL \
    --max-concurrency 16 \
    --method real_gpu_$MODEL \
    --notes "one-shot Colab run on T4"

## 9. Part C — load test + GPU monitor

In [ ]:
import subprocess

# Start GPU sampler in the background.
gpu_proc = subprocess.Popen(
    ["uv", "run", "python", "-m", "perf.gpu_monitor",
     "--output", "results/perf/gpu.csv",
     "--interval", "1", "--duration", "180"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Load test.
!uv run python -m perf.load_test \
    --num-requests 200 --concurrency 16 --mode stream \
    --output results/perf/metrics.csv

# Summarise.
!uv run python -m perf.metrics --input results/perf/metrics.csv --summary results/perf/summary.csv

gpu_proc.terminate()
try:
    gpu_proc.wait(timeout=5)
except subprocess.TimeoutExpired:
    gpu_proc.kill()

!cat results/perf/summary.csv

## 10. Part E — full HellaSwag ablation ladder

One row per variant, paired-bootstrap CI vs baseline, all rows landing in
`results/result-log.csv` and `docs/improvement-log.md`.

In [ ]:
os.environ["N_EVAL"] = str(N_EVAL)
os.environ["N_POOL"] = "500"
os.environ["MAX_CONCURRENCY"] = "16"

# Paranoia: nuke any residual Part E data before running. improve/eval.sh
# already forces re-prep, but if this cell is re-run after a failed attempt
# leftover files could still be sitting around.
!rm -rf results/improve

!bash improve/eval.sh

# Sanity gate right here -- fail fast if the ablation scored synthetic data.
import json
from pathlib import Path

eval_jsonl = Path("results/improve/hellaswag_eval.jsonl")
assert eval_jsonl.exists(), "improve/eval.sh didn't produce hellaswag_eval.jsonl"
first_ind = json.loads(eval_jsonl.read_text().splitlines()[0]).get("ind", "")
assert not str(first_ind).startswith("syn-"), (
    f"FAIL: Part E scored synthetic data (first ind={first_ind!r}). "
    "Restart the runtime from scratch -- Cell 3's fresh-clone guard will "
    "rule out the most common cause."
)
print(f"OK: Part E scored real HellaSwag (first ind={first_ind!r})")

# Also require all 7 variant accuracies are strictly between 0 and 1 --
# synthetic data pegs fewshot_*_5 at exactly 1.0000.
import glob
for f in sorted(glob.glob("results/improve/*.json")):
    if "hellaswag" in f:
        continue
    d = json.loads(Path(f).read_text())
    name = f.split("/")[-1][:-5]
    assert 0.0 < d["accuracy"] < 1.0, (
        f"{name} has accuracy={d['accuracy']} -- 0.0 or 1.0 is the synthetic-data fingerprint"
    )
    print(f"  {name:25s} acc={d['accuracy']:.4f} n={d['n']}")

## 11. Determinism check (Part D)

In [ ]:
!uv run python -m guardrails.validate determinism \
    "The capital of France is" --n-runs 5 --max-tokens 16 --seed 42 \
    --report results/determinism_report.json
!cat results/determinism_report.json

## 12. Inspect the final tables

In [ ]:
print("=== results/summary.md ===")
!cat results/summary.md
print()
print("=== results/result-log.md ===")
!cat results/result-log.md
print()
print("=== docs/improvement-log.md (tail) ===")
!tail -n 80 docs/improvement-log.md

## 13. Shut vLLM down and package results

In [ ]:
import signal
serve_proc.send_signal(signal.SIGTERM)
try:
    serve_proc.wait(timeout=30)
except subprocess.TimeoutExpired:
    serve_proc.kill()
print("vLLM stopped.")

!tar czf /tmp/llmeval_results.tgz \
    results/ \
    docs/improvement-log.md \
    improve/report.md \
    vllm_serve.log

!ls -lh /tmp/llmeval_results.tgz

## 14. Download the tarball to your laptop

Unpack it into your repo checkout, commit, and submit.

In [ ]:
from google.colab import files
files.download("/tmp/llmeval_results.tgz")

### Back on your laptop

```bash
cd path/to/LLM-Evaluation-Pipeline
tar xzf ~/Downloads/llmeval_results.tgz
git add results/ docs/improvement-log.md improve/report.md
git status
git commit -m "chore: real numbers from Colab T4 run on <model>"
```

Your `improve/report.md` results table now has real numbers; submit the
repo per the problem-statement submission instructions.